In [1]:
import warnings
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import TOSICA
import scipy.stats as stats
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import torch

torch_device = torch.device('cuda:1')  # 使用第二块 GPU
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

plt.rcParams['figure.figsize'] = [6, 4.5]
plt.rcParams["savefig.dpi"] = 300

/data/jiangjunyao/miniconda3/envs/TOSICA/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import time

import psutil

import os

import pandas as pd

from sklearn.metrics import accuracy_score


def train_tosica(train_data, test_data, label_name):
    """
    使用 TOSICA 训练模型并计算测试集准确性，同时记录运行时间和内存使用情况。
    
    参数:
    - train_data: 训练集 (AnnData 对象)
    - test_data: 测试集 (AnnData 对象)
    - label_name: 标签列名

    
    返回:
    - test_accuracy: 测试集的准确性

    - runtime: 运行时间（秒）
    - memory_usage_gb: 内存使用（GB）
    - predicted_labels_df: 每个 cell 的预测标签 (DataFrame 格式)
    """
    # 记录开始时间和初始内存使用

    start_time = time.time()
    process = psutil.Process(os.getpid())
    initial_memory = process.memory_info().rss / (1024 ** 3)  # 初始内存使用 (GB)


    # 使用 TOSICA 进行细胞类型注释
    if train_data.var_names[0].isupper():
        gmt1 = 'human_gobp'
    else:
        gmt1 = 'mouse_gobp'

    TOSICA.train(train_data, gmt_path= gmt1, label_name='celltype',epochs=50,project='aa2')
    new_adata = TOSICA.pre(test_data,project='aa2',model_weight_path='./aa2/model-49.pth')
    # 对测试集进行预测

    predicted_labels = new_adata.obs['Prediction'].values
    true_labels = test_data.obs[label_name].values

    # 计算测试集准确性

    test_accuracy = accuracy_score(true_labels, predicted_labels)

    # 创建预测标签的 DataFrame

    predicted_labels_df = pd.DataFrame({
        "cell_id": test_data.obs_names,
        "true_label": true_labels,
        "predicted_label": predicted_labels

    })

    # 记录结束时间和最终内存使用

    end_time = time.time()
    final_memory = process.memory_info().rss / (1024 ** 3)  # 最终内存使用 (GB)

    # 计算运行时间和内存使用

    runtime = end_time - start_time

    memory_usage_gb = final_memory - initial_memory

    return test_accuracy, runtime, memory_usage_gb, predicted_labels_df

In [5]:
split_data_path = "/data/jiangjunyao/AEGAS data/celltype annotation/AEGAS_anno/intra_fivefold_split/"
outdir = '/data/jiangjunyao/AEGAS data/celltype annotation/anno_result/predict_result/tosica'
h5ad_files = [f for f in os.listdir(split_data_path) if os.path.isdir(os.path.join(split_data_path, f))]

output_results = []

# 遍历每个数据集

#for dataset in h5ad_files:
for dataset in [
               'ProksNM_12_humanembryo_fold_1','ProksNM_12_humanembryo_fold_2','ProksNM_12_humanembryo_fold_3','ProksNM_12_humanembryo_fold_4','ProksNM_12_humanembryo_fold_5']:    
    print(f"处理数据集：{dataset}")
    dataset_path = split_data_path +dataset

    fold_accuracies = []
    fold_runtimes = []
    fold_memories = []

    # 读取训练集和测试集
    train_data = sc.read_h5ad(dataset_path+"/train.h5ad")
    test_data = sc.read_h5ad(dataset_path+"/test.h5ad")

    # 确保数据中有标签列

    label_name = "celltype"  # 假设标签列为 "cell_type"，请根据实际情况修改

    if label_name not in train_data.obs.columns or label_name not in test_data.obs.columns:
        raise ValueError(f"missing label:  '{label_name}'")

    # 使用 scANVI 训练并计算测试准确性、运行时间和内存使用

    test_accuracy, runtime, memory_usage_gb,result = train_tosica(train_data, test_data, label_name)
    fold_accuracies.append(test_accuracy)
    fold_runtimes.append(runtime)
    fold_memories.append(memory_usage_gb)
    result.to_csv(outdir+dataset+'.csv')

    # 保存每个数据集的结果

    output_results.append({
        "dataset": dataset,
        "fold_accuracies": fold_accuracies,
        "fold_runtimes": fold_runtimes,
        "fold_memories": fold_memories,
        "mean_accuracy": np.mean(fold_accuracies),
        "mean_runtime": np.mean(fold_runtimes),
        "mean_memory": np.mean(fold_memories)
    })

# 创建 DataFrame 并保存结果

results_df = pd.DataFrame(output_results)
results_df = results_df[["dataset", "mean_accuracy", "mean_runtime", "mean_memory"]]  # 选择关键列


处理数据集：ProksNM_12_humanembryo_fold_1
cuda:0
Mask loaded!
Model builded!


[valid epoch 49] loss: 0.066, acc: 0.989: 100%|██████████| 455/455 [00:00<00:00, 595.78it/s]


Training finished!
cuda:0
0


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [18]:
results_df = pd.DataFrame(output_results)
results_df = results_df[["dataset", "mean_accuracy", "mean_runtime", "mean_memory"]]  # 选择关键列

print(results_df)
results_df.to_csv('/data/jiangjunyao/AEGAS data/celltype annotation/AEGAS_anno/anno_summary/tosica_intra.csv')

                            dataset  mean_accuracy  mean_runtime  mean_memory
0  ZhangCell_10_colon_cancer_fold_1       0.793219   1744.272975    -0.000683
1  ZhangCell_10_colon_cancer_fold_2       0.806590   1596.422443    -0.000614
2  ZhangCell_10_colon_cancer_fold_3       0.807545   1675.289272    -0.000683
3  ZhangCell_10_colon_cancer_fold_4       0.758242   1487.727118     0.001850
4  ZhangCell_10_colon_cancer_fold_5       0.816054   1385.646863    -0.031944
